In [ ]:
import json
import time
import math
import os
import re
import string
from google import genai
from dotenv import load_dotenv

load_dotenv() # creat a .env file with GEMINI_API_KEY=your_api_key (gitignored the env file for security)
api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

# load dataset
public_data = [json.loads(line) for line in open("./data/public.jsonl")]

n_mcq  = sum(bool(d.get("options")) for d in public_data)
n_free = sum(not d.get("options")   for d in public_data)
print(f"Loaded {len(public_data)} questions  ({n_mcq} MCQ, {n_free} FRQ)")

In [ ]:
# prompts for free response and MCQ problems
SYSTEM_PROMPT_FRQ = (
    "Be mathematically correct. "
    "Be brief but complete. "
    "Show visible reasoning. "
    "Do not use markdown formatting such as bold text or bullet points. "
    'Do not write step labels like "Step 1" or "Identify". '
    "Write the solution as a clean mathematical derivation. "
    "End with exactly one final answer in \\boxed{}. "
    "If there are multiple answers, put them in one \\boxed{} separated by commas. "
    "The final boxed answer must exactly match the known correct answer. "
    "Do not mention the known answer explicitly. "
    "Preserve 1e-8 precision if needed."
)

SYSTEM_PROMPT_MCQ = (
    "Be mathematically correct. "
    "Be brief but complete. "
    "Show visible reasoning. "
    "Do not use markdown formatting such as bold text or bullet points. "
    'Do not write step labels like "Step 1" or "Identify". '
    "Write the solution as a clean mathematical derivation. "
    "Use the answer choices to determine the correct option. "
    "End with exactly one final answer in the form \\boxed{<letter>}. "
    "The final boxed answer must exactly match the known correct answer. "
    "Do not mention the known correct option explicitly."
)


FRQ_TEMPLATE = """Problem:
{question}

Known correct answer:
{answer}
"""

MCQ_TEMPLATE = """Problem:
{question}

Options:
{options}

Known correct option:
{answer}
"""

In [ ]:
from typing import Optional

def normalize_answer(row):
    # normalizes answer to string, changes ['1', '2'] to "1, 2" for example
    ans = row["answer"]

    if isinstance(ans, list):
        ans = [str(x).strip() for x in ans]
        return ", ".join(ans)

    return str(ans).strip()

def build_prompt(row):
    question = row["question"]
    answer   = normalize_answer(row)
    options  = row.get("options")

    return build_prompt_internal(question, options, answer)

def build_prompt_internal(question: str, options: Optional[list], answer: str):
    """Return (system_prompt, User_prompt(question, answer))"""
    
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return (SYSTEM_PROMPT_MCQ, MCQ_TEMPLATE.format(question=question, options=opts_text, answer=answer))
    return (SYSTEM_PROMPT_FRQ, FRQ_TEMPLATE.format(question=question, answer=answer))

In [ ]:

from google.genai import errors

def generate_solution(system_prompt, user_prompt):
    prompt = f"""{system_prompt} {user_prompt}"""

    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=prompt,
    )

    return response.text

def generate_retry(system_prompt, user_prompt):
    retries = 5
    for i in range(retries):
        try:
            return generate_solution(system_prompt, user_prompt)
        except errors.APIError as e:
            print(f"API error on attempt {i+1}/{retries}: {e}")
            time.sleep(60*(1.7*i)) # wait before retrying
    raise Exception("Failed to generate solution after multiple attempts")


results = []

for row in public_data[:50]:
    system_prompt, user_prompt = build_prompt(row)
    solution = generate_retry(system_prompt, user_prompt)

    example = {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": solution},
        ]
    }
    time.sleep(20) # to avoid rate limits
    results.append(example)

In [ ]:
with open("./data/llm_train.jsonl", "w", encoding="utf-8") as f:
    for example in results:
        f.write(json.dumps(example, ensure_ascii=False) + "\n")